# Test a hybrid next-concept checkpoint

Loads `hybrid.pt` from the training notebook and tests it. Nothing here updates weights.

The checkpoint was trained on CodeSearchNet Python functions. This notebook does not train. It loads the **test** split, which the training notebook never reads and which comes from other repositories, and asks three questions:

1. Can the residual quantizer reconstruct those tokens?
2. If the coarser codes are right, can the generator predict the next level?
3. Does one teacher repair pass make a free draft more like the teacher, and how many forwards does that cost?

A short training run can fail the quality checks. That is a result, not a notebook error. The repair-rule check should pass even then: it only tests that low-confidence tokens are the ones that change.

Runtime → Change runtime type → **T4 GPU**. The teacher is loaded in fp16. A 0.5B checkpoint stays well under 16 GB. A 1.5B checkpoint fits. 3B and larger are refused.


In [ ]:
%pip install -q "transformers>=5.3,<6" "datasets>=3" "huggingface-hub>=1.32,<2" accelerate sentencepiece protobuf


In [ ]:
import gc
import math
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

CHECKPOINT_CANDIDATES = [
    Path("/content/ncm-hybrid/hybrid.pt"),
    Path("/content/drive/MyDrive/ncm-hybrid/hybrid.pt"),
    Path("hybrid.pt"),
]
N_TEST = 256
BENCH_BATCH = 4
PREFIX = 32
SEED = 99
DATASET_ID = "code-search-net/code_search_net"
DATASET_CONFIG = "python"

if not torch.cuda.is_available():
    raise SystemExit("No GPU. In Colab: Runtime → Change runtime type → T4 GPU, then rerun.")
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3
print(f"GPU {props.name}  {vram_gb:.1f} GB")
if vram_gb < 12:
    raise SystemExit(f"This test wants a 16 GB T4. This GPU reports {vram_gb:.1f} GB.")

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")


In [ ]:
class MultiScaleResidualVQ(nn.Module):
    """Residual vector quantizer. One codebook per level. A small transformer, not a linear map, reads the reconstruction back into tokens."""

    def __init__(self, vocab: int, dim: int, codebook: int, levels: list[int], n_dec: int = 2, n_head: int = 6):
        super().__init__()
        self.levels = levels
        self.dim = dim
        self.embed = nn.Embedding(vocab, dim)
        self.codebooks = nn.ParameterList(
            [nn.Parameter(torch.randn(codebook, dim) * 0.02) for _ in range(len(levels))]
        )
        self.dec_blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(dim, n_head, dim * 4, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
                for _ in range(n_dec)
            ]
        )
        self.dec_norm = nn.LayerNorm(dim)
        self.dec_head = nn.Linear(dim, vocab, bias=False)
        self.dec_head.weight = self.embed.weight
        nn.init.normal_(self.embed.weight, std=0.02)

    def _bins(self, t: int, length: int) -> list[tuple[int, int]]:
        bins = []
        for i in range(length):
            start = (i * t) // length
            end = min(t, max(start + 1, ((i + 1) * t) // length))
            bins.append((start, end))
        return bins

    def _pool(self, residual: torch.Tensor, length: int) -> tuple[torch.Tensor, list[tuple[int, int]]]:
        bins = self._bins(residual.shape[1], length)
        chunks = [residual[:, start:end].mean(dim=1) for start, end in bins]
        return torch.stack(chunks, dim=1), bins

    def _upsample(self, codes: torch.Tensor, bins: list[tuple[int, int]], t: int) -> torch.Tensor:
        out = codes.new_zeros(codes.shape[0], t, codes.shape[-1])
        for i, (start, end) in enumerate(bins):
            out[:, start:end] = codes[:, i : i + 1]
        return out

    def quantize(self, pooled: torch.Tensor, level: int) -> tuple[torch.Tensor, torch.Tensor]:
        book = self.codebooks[level]
        # argmin ||p - c||^2 == argmax p·c - 0.5 ||c||^2
        score = pooled @ book.t() - 0.5 * book.pow(2).sum(-1)
        idx = score.argmax(-1)
        return idx, book[idx]

    def encode(self, tokens: torch.Tensor) -> list[torch.Tensor]:
        indices, _before = self.contexts(tokens)
        return indices

    def contexts(self, tokens: torch.Tensor) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        """Codes for every level, and the cumulative reconstruction from before that level."""
        residual = self.embed(tokens)
        recon = torch.zeros_like(residual)
        indices = []
        before = []
        for li, length in enumerate(self.levels):
            before.append(recon)
            pooled, bins = self._pool(residual, length)
            idx, q = self.quantize(pooled, li)
            up = self._upsample(q, bins, tokens.shape[1])
            residual = residual - up
            recon = recon + up
            indices.append(idx)
        return indices, before

    def add_level(self, recon: torch.Tensor, idx: torch.Tensor, level: int) -> torch.Tensor:
        bins = self._bins(recon.shape[1], self.levels[level])
        return recon + self._upsample(self.codebooks[level][idx], bins, recon.shape[1])

    def _decode(self, recon: torch.Tensor) -> torch.Tensor:
        h = recon
        for block in self.dec_blocks:
            h = block(h)
        return self.dec_head(self.dec_norm(h))

    def logits_from_codes(self, indices: list[torch.Tensor]) -> torch.Tensor:
        t = self.levels[-1]
        recon = self.embed.weight.new_zeros(indices[0].shape[0], t, self.dim)
        for li, idx in enumerate(indices):
            recon = self.add_level(recon, idx, li)
        return self._decode(recon)

    def reconstruction_loss(self, tokens: torch.Tensor, keep: int | None = None) -> tuple[torch.Tensor, torch.Tensor]:
        residual = self.embed(tokens)
        recon = torch.zeros_like(residual)
        commit = tokens.new_zeros((), dtype=torch.float)
        book = tokens.new_zeros((), dtype=torch.float)
        n = keep or len(self.levels)
        for li in range(n):
            pooled, bins = self._pool(residual, self.levels[li])
            _idx, q = self.quantize(pooled, li)
            q_st = pooled + (q - pooled).detach()
            up = self._upsample(q_st, bins, tokens.shape[1])
            residual = residual - up
            recon = recon + up
            commit = commit + (pooled - q.detach()).pow(2).mean()
            book = book + (q - pooled.detach()).pow(2).mean()
        logits = self._decode(recon)
        ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tokens.reshape(-1))
        acc = (logits.argmax(-1) == tokens).float().mean()
        return ce + 0.25 * (commit / n) + book / n, acc


class NextConcept(nn.Module):
    """Finer levels are parallel. The coarsest level is a causal next-code model.

    Position t of the coarsest level predicts that code from codes 0..t-1 only.
    A future code is never an input to its own prediction.
    """

    def __init__(self, vq: MultiScaleResidualVQ, dim: int, n_layer: int, n_head: int):
        super().__init__()
        self.levels = list(vq.levels)
        self.vq = vq
        codebook = vq.codebooks[0].shape[0]
        self.in_proj = nn.Linear(vq.dim, dim)
        self.pos = nn.ParameterList([nn.Parameter(torch.randn(length, dim) * 0.02) for length in self.levels])
        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(dim, n_head, dim * 4, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
                for _ in range(n_layer)
            ]
        )
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, codebook)
        coarse_len = self.levels[0]
        self.coarse_in = nn.Linear(vq.dim, dim)
        self.coarse_bos = nn.Parameter(torch.zeros(1, 1, dim))
        self.coarse_pos = nn.Parameter(torch.randn(coarse_len, dim) * 0.02)
        self.coarse_blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(dim, n_head, dim * 4, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
                for _ in range(4)
            ]
        )
        self.coarse_norm = nn.LayerNorm(dim)
        self.coarse_head = nn.Linear(dim, codebook)

    def forward_level(self, recon: torch.Tensor, level: int) -> torch.Tensor:
        if level == 0:
            raise RuntimeError("the coarsest level is causal; call forward_coarse")
        pooled, _bins = self.vq._pool(recon, self.levels[level])
        x = self.in_proj(pooled) + self.pos[level]
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

    def forward_coarse(self, codes: torch.Tensor) -> torch.Tensor:
        """codes is (batch, level_length). Logits at t depend only on codes before t."""
        emb = self.coarse_in(self.vq.codebooks[0][codes])
        bos = self.coarse_bos.expand(codes.shape[0], 1, -1)
        x = torch.cat([bos, emb[:, :-1]], dim=1) + self.coarse_pos
        length = codes.shape[1]
        mask = torch.triu(torch.full((length, length), float("-inf"), device=x.device), diagonal=1)
        for block in self.coarse_blocks:
            x = block(x, src_mask=mask)
        return self.coarse_head(self.coarse_norm(x))


def init_from_teacher(vq: MultiScaleResidualVQ, teacher_embed: torch.Tensor) -> None:
    with torch.no_grad():
        src = teacher_embed.float()
        rows = min(vq.embed.weight.shape[0], src.shape[0])
        if src.shape[1] == vq.dim:
            copied = src[:rows]
        else:
            # First 256 coordinates of an 896-d table are not a projection.
            gen = torch.Generator().manual_seed(0)
            proj = torch.randn(src.shape[1], vq.dim, generator=gen) / math.sqrt(src.shape[1])
            copied = src[:rows] @ proj
        vq.embed.weight[:rows].copy_(copied)
        for book in vq.codebooks:
            pick = torch.randint(0, rows, (book.shape[0],))
            book.copy_(vq.embed.weight[pick])


@torch.no_grad()
def generate_ncm(
    gen: NextConcept,
    vq: MultiScaleResidualVQ,
    unigram: torch.Tensor,
    batch: int | None = None,
    prefixes: torch.Tensor | None = None,
) -> torch.Tensor:
    """Draft a window. The prefix keeps its encoded codes.

    The coarsest future codes are predicted one at a time, each from the codes
    before it. Finer levels then run in parallel on that sketch. The returned
    tokens start with the prefix.
    """
    if prefixes is not None:
        batch = prefixes.shape[0]
        known_idx, known_mask = _prefix_codes(vq, prefixes)
    else:
        known_idx = known_mask = None
        if batch is None:
            raise ValueError("pass batch or prefixes")
    device = vq.embed.weight.device
    recon = torch.zeros(batch, vq.levels[-1], vq.dim, device=device, dtype=vq.embed.weight.dtype)
    if known_idx is not None:
        coarse = known_idx[0].clone()
        for t in range(coarse.shape[1]):
            if bool(known_mask[0][:, t].all()):
                continue
            coarse[:, t] = gen.forward_coarse(coarse)[:, t].argmax(-1)
    else:
        coarse = torch.zeros(batch, vq.levels[0], dtype=torch.long, device=device)
        for t in range(coarse.shape[1]):
            coarse[:, t] = gen.forward_coarse(coarse)[:, t].argmax(-1)
    codes = [coarse]
    recon = vq.add_level(recon, coarse, 0)
    for li in range(1, len(vq.levels)):
        pred = gen.forward_level(recon, li).argmax(-1)
        if known_mask is not None:
            pred = torch.where(known_mask[li], known_idx[li], pred)
        codes.append(pred)
        recon = vq.add_level(recon, pred, li)
    tokens = vq.logits_from_codes(codes).argmax(-1)
    if prefixes is not None:
        tokens = tokens.clone()
        tokens[:, : prefixes.shape[1]] = prefixes
    return tokens


@torch.no_grad()
def _prefix_codes(vq: MultiScaleResidualVQ, prefixes: torch.Tensor):
    residual = vq.embed(prefixes)
    known_idx = []
    known_mask = []
    p = prefixes.shape[1]
    for li, length in enumerate(vq.levels):
        bins = vq._bins(vq.levels[-1], length)
        idx = torch.zeros(prefixes.shape[0], length, dtype=torch.long, device=prefixes.device)
        mask = torch.zeros(prefixes.shape[0], length, dtype=torch.bool, device=prefixes.device)
        for i, (start, end) in enumerate(bins):
            if end <= p and end > start:
                code, q = vq.quantize(residual[:, start:end].mean(dim=1, keepdim=True), li)
                idx[:, i] = code[:, 0]
                mask[:, i] = True
                residual[:, start:end] = residual[:, start:end] - q
        known_idx.append(idx)
        known_mask.append(mask)
    return known_idx, known_mask


@torch.no_grad()
def hybrid_repair(draft: torch.Tensor, teacher, tau: float, prefix: int = 0) -> tuple[torch.Tensor, float]:
    """One causal teacher forward. Low-confidence draft tokens take the teacher's prediction.

    Logits at position t-1 predict token t, so a replacement does not re-enter this forward.
    Tokens inside `prefix` are the given function start and are not rewritten.
    The rate is the fraction rewritten in the continuation.
    """
    logits = teacher(draft).logits
    pred = logits.argmax(-1)
    probs = logits.softmax(-1)
    conf = probs[:, :-1].gather(-1, draft[:, 1:].unsqueeze(-1)).squeeze(-1)
    replace = conf < tau
    if prefix > 1:
        replace[:, : prefix - 1] = False
    hybrid = draft.clone()
    hybrid[:, 1:] = torch.where(replace, pred[:, :-1], draft[:, 1:])
    tail = replace if prefix <= 1 else replace[:, prefix - 1 :]
    return hybrid, tail.float().mean().item()


def teacher_nll(teacher, tokens: torch.Tensor) -> float:
    logits = teacher(tokens).logits
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1)).item()


def cuda_ms(fn):
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    torch.cuda.reset_peak_memory_stats()
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    return out, start.elapsed_time(end), torch.cuda.max_memory_allocated() / 1024**3


## Load the checkpoint

Looks in `/content/ncm-hybrid/hybrid.pt`, then Drive, then the working directory. If none of those exist, the next cell opens a file picker. Mount Drive yourself before running if the file lives there. This cell does not mount anything.


In [ ]:
checkpoint_path = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), None)
if checkpoint_path is None:
    try:
        from google.colab import files
        print("No hybrid.pt found. Upload the file saved by the training notebook.")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No file uploaded.")
        name = next(iter(uploaded))
        checkpoint_path = Path("/content") / name
    except ImportError:
        raise SystemExit("Put hybrid.pt at /content/ncm-hybrid/hybrid.pt and rerun this cell.")

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
needed = ["model", "levels", "codebook", "vq_dim", "gen_dim", "gen_layers", "gen_heads", "tau", "vq", "generator", "unigram"]
missing = [key for key in needed if key not in ckpt]
if missing:
    raise SystemExit(f"{checkpoint_path} is missing {missing}. It is not a training-notebook checkpoint.")

model_id = ckpt["model"]
low = model_id.lower()
if any(tag in low for tag in ["7b", "8b", "9b", "14b", "32b", "70b", "72b", "4b", "3b"]):
    raise SystemExit(f"{model_id} does not fit a free 16 GB T4. Retrain the checkpoint from a 0.5B or 1.5B teacher.")

levels = [int(x) for x in ckpt["levels"]]
vocab = int(ckpt["vq"]["embed.weight"].shape[0])
codebook = int(ckpt["codebook"])
if int(ckpt["vq"]["codebooks.0"].shape[0]) != codebook:
    raise SystemExit("codebook size in the file does not match the codebook tensor")

n_dec = int(ckpt.get("dec_layers", 2))
n_head = int(ckpt["gen_heads"])
PREFIX = int(ckpt.get("prefix", PREFIX))
vq = MultiScaleResidualVQ(vocab, int(ckpt["vq_dim"]), codebook, levels, n_dec=n_dec, n_head=n_head).to(device)
gen = NextConcept(vq, int(ckpt["gen_dim"]), int(ckpt["gen_layers"]), n_head).to(device)
if not any(key.startswith("coarse_") for key in ckpt["generator"]):
    raise SystemExit("This checkpoint predicts every future coarse code at once. Use a causal-next-code checkpoint.")
gen.load_state_dict(ckpt["generator"], strict=True)
unigram = ckpt["unigram"].to(device).float()
unigram = unigram / unigram.sum()
vq.eval()
gen.eval()
tau = float(ckpt["tau"])
context = levels[-1]
print(f"loaded {checkpoint_path}")
print(f"teacher id {model_id} | context {context} | prefix {PREFIX} | levels {len(levels)} | tau {tau}")
print(f"allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## Held-out text

CodeSearchNet's Python test split. The training notebook uses the training split and scores the validation split. Repositories do not cross splits. Windows are real functions, not teacher samples of a prompt list.


In [ ]:
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if int(tokenizer.vocab_size) != vocab:
    raise SystemExit(f"tokenizer vocab {tokenizer.vocab_size} does not match checkpoint vocab {vocab}")

teacher = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, low_cpu_mem_usage=True)
teacher.to(device)
teacher.eval()
print(f"teacher resident {torch.cuda.memory_allocated()/1024**3:.2f} GB")


def code_windows(split: str, limit: int, seed: int) -> torch.Tensor:
    ds = load_dataset(DATASET_ID, DATASET_CONFIG, split=split, streaming=True)
    ds = ds.shuffle(buffer_size=2048, seed=seed)
    need = limit * context
    ids: list[int] = []
    kept = 0
    bar = tqdm(total=limit, desc=f"{split} windows", unit="win", mininterval=1, dynamic_ncols=True)
    for row in ds:
        text = (row.get("whole_func_string") or "").replace("\r\n", "\n").strip()
        if len(text) < 40 or "def " not in text:
            continue
        piece = tokenizer(text + "\n", add_special_tokens=False)["input_ids"]
        if len(piece) < 16 or len(piece) > 1024:
            continue
        ids.extend(piece)
        kept += 1
        bar.n = min(limit, len(ids) // context)
        bar.set_postfix(functions=kept, refresh=False)
        bar.refresh()
        if len(ids) >= need + context:
            break
    bar.close()
    usable = (len(ids) // context) * context
    if usable < need:
        raise SystemExit(f"{split} produced {usable // context} windows, need {limit}.")
    windows = torch.tensor(ids[:usable], dtype=torch.long).view(-1, context)
    order = torch.randperm(windows.shape[0], generator=torch.Generator().manual_seed(seed))
    chosen = windows[order[:limit]].contiguous()
    print(f"{split}: {kept} functions, {tuple(chosen.shape)}")
    return chosen


@torch.no_grad()
def teacher_from_prefixes(prefixes: torch.Tensor, sample: bool) -> torch.Tensor:
    attention = torch.ones_like(prefixes)
    new = context - prefixes.shape[1]
    kwargs = {"do_sample": True, "temperature": 0.9, "top_p": 0.95} if sample else {"do_sample": False}
    out = teacher.generate(
        input_ids=prefixes,
        attention_mask=attention,
        max_new_tokens=new,
        pad_token_id=tokenizer.pad_token_id,
        **kwargs,
    )
    full = out[:, :context]
    if full.shape[1] < context:
        full = torch.nn.functional.pad(full, (0, context - full.shape[1]), value=tokenizer.pad_token_id)
    return full


In [ ]:
@torch.no_grad()
def reconstruction_accuracy(tokens):
    correct = 0
    total = 0
    for start in range(0, tokens.shape[0], 4):
        batch = tokens[start : start + 4]
        pred = vq.logits_from_codes(vq.encode(batch)).argmax(-1)
        correct += (pred == batch).sum().item()
        total += batch.numel()
    return correct / max(1, total)


@torch.no_grad()
def concept_accuracy(tokens):
    per_level = []
    correct = 0
    total = 0
    for start in range(0, tokens.shape[0], 4):
        batch = tokens[start : start + 4]
        codes, recons = vq.contexts(batch)
        for li in range(len(levels)):
            if li == 0:
                pred = gen.forward_coarse(codes[0]).argmax(-1)
            else:
                pred = gen.forward_level(recons[li], li).argmax(-1)
            hit = (pred == codes[li]).sum().item()
            if len(per_level) <= li:
                per_level.append([0, 0])
            per_level[li][0] += hit
            per_level[li][1] += codes[li].numel()
            correct += hit
            total += codes[li].numel()
    return correct / max(1, total), [hit / count for hit, count in per_level]


@torch.no_grad()
def cascade_accuracy(tokens):
    correct = 0
    total = 0
    for start in range(0, tokens.shape[0], 4):
        batch = tokens[start : start + 4]
        recon = torch.zeros(batch.shape[0], batch.shape[1], vq.dim, device=batch.device)
        coarse = torch.zeros(batch.shape[0], levels[0], dtype=torch.long, device=batch.device)
        for t in range(levels[0]):
            coarse[:, t] = gen.forward_coarse(coarse)[:, t].argmax(-1)
        codes = [coarse]
        recon = vq.add_level(recon, coarse, 0)
        for li in range(1, len(levels)):
            idx = gen.forward_level(recon, li).argmax(-1)
            codes.append(idx)
            recon = vq.add_level(recon, idx, li)
        pred = vq.logits_from_codes(codes).argmax(-1)
        correct += (pred == batch).sum().item()
        total += batch.numel()
    return correct / max(1, total)


def repair_rule_holds(draft, hybrid, teacher_model, tau_value, prefix=0):
    logits = teacher_model(draft).logits
    pred = logits.argmax(-1)
    conf = logits.softmax(-1)[:, :-1].gather(-1, draft[:, 1:].unsqueeze(-1)).squeeze(-1)
    replace = conf < tau_value
    if prefix > 1:
        replace[:, : prefix - 1] = False
    kept = hybrid[:, 1:][~replace]
    kept_draft = draft[:, 1:][~replace]
    changed = hybrid[:, 1:][replace]
    changed_teacher = pred[:, :-1][replace]
    head = max(prefix, 1)
    same_prefix = torch.equal(hybrid[:, :head], draft[:, :head])
    kept_ok = kept.numel() == 0 or torch.equal(kept, kept_draft)
    changed_ok = changed.numel() == 0 or torch.equal(changed, changed_teacher)
    return same_prefix and kept_ok and changed_ok, replace.float().mean().item()


## Run the tests

Reconstruction and concept accuracy use CodeSearchNet test windows. The speed table continues the first `PREFIX` tokens of those windows. The concept draft encodes the bins inside that prefix and predicts the rest. The hybrid pass does not rewrite the prefix. Negative log-likelihood is the teacher's score of the finished window. A lower hybrid number is not a better function when the repair rate is high: the inserted tokens are the teacher's local mode, often whitespace.


In [ ]:
held = code_windows("test", N_TEST, SEED).to(device)
print(tokenizer.decode(held[0].tolist(), skip_special_tokens=True)[:500])

recon = reconstruction_accuracy(held)
concept, per_level = concept_accuracy(held)
cascade = cascade_accuracy(held)
chance = 1.0 / vocab
print(f"tokenizer reconstruction {recon:.3%}   chance {chance:.4%}")
print(f"next-level code accuracy {concept:.3%}   (conditioned on the true reconstruction so far)")
print(f"cascade from the coarsest code {cascade:.3%}")
print("per level:", " ".join(f"L{i}:{acc:.2f}" for i, acc in enumerate(per_level)))

with torch.no_grad():
    teacher_tokens, teacher_ms, teacher_peak = cuda_ms(lambda: teacher_from_prefixes(held[:BENCH_BATCH, :PREFIX], sample=False))
    teacher_nll_value = teacher_nll(teacher, teacher_tokens)
    draft, ncm_ms, ncm_peak = cuda_ms(lambda: generate_ncm(gen, vq, unigram, prefixes=held[:BENCH_BATCH, :PREFIX]))
    ncm_nll_value = teacher_nll(teacher, draft)

    calls = {"n": 0}
    original_forward = teacher.forward

    def counting_forward(*args, **kwargs):
        calls["n"] += 1
        return original_forward(*args, **kwargs)

    teacher.forward = counting_forward
    try:
        (hybrid, reported_rate), repair_ms, repair_peak = cuda_ms(lambda: hybrid_repair(draft, teacher, tau, prefix=PREFIX))
        forward_calls = calls["n"]
        rule_ok, rate = repair_rule_holds(draft, hybrid, teacher, tau, prefix=PREFIX)
    finally:
        teacher.forward = original_forward
    hybrid_nll_value = teacher_nll(teacher, hybrid)

rows = [
    ("teacher", teacher_nll_value, context - PREFIX, teacher_ms / BENCH_BATCH, teacher_peak, "greedy continuation"),
    ("next-concept", ncm_nll_value, len(levels), ncm_ms / BENCH_BATCH, ncm_peak, "same prefix"),
    ("hybrid", hybrid_nll_value, len(levels) + 1, (ncm_ms + repair_ms) / BENCH_BATCH, max(ncm_peak, repair_peak), f"repair rate {rate:.0%}"),
]
print()
print(f"{'writer':16} {'NLL':>8} {'forwards':>10} {'ms/seq':>10} {'peak GB':>8}  note")
for name, nll, forwards, ms, peak, note in rows:
    print(f"{name:16} {nll:8.3f} {forwards:10d} {ms:10.1f} {peak:8.2f}  {note}")

checks = [
    ("repair leaves token 0 unchanged and only replaces low-confidence tokens", rule_ok),
    ("repair is a single teacher forward", forward_calls == 1),
    ("hybrid uses fewer forwards than the teacher", len(levels) + 1 < context),
    ("hybrid NLL is lower than the raw draft", hybrid_nll_value < ncm_nll_value),
]
print()
failed = 0
for label, ok in checks:
    print(("PASS  " if ok else "FAIL  ") + label)
    failed += not ok
print()
if recon < 0.05:
    print(f"NOTE  tokenizer reconstruction is {recon:.3%}. The quantizer has not learned token identity on this checkpoint.")
else:
    print(f"NOTE  tokenizer reconstruction is {recon:.3%}. Chance is {chance:.4%}.")
if rate > 0.9:
    print(f"NOTE  repair rate is {rate:.0%}. The teacher is rewriting almost the whole draft, so the hybrid is mostly the teacher.")
elif rate < 0.02:
    print(f"NOTE  repair rate is {rate:.0%}. The threshold barely fired. The hybrid is the draft.")
print("failed checks:", failed)

print("\n--- teacher ---")
print(tokenizer.decode(teacher_tokens[0], skip_special_tokens=True)[:400])
print("\n--- next-concept draft ---")
print(tokenizer.decode(draft[0], skip_special_tokens=True)[:400])
print("\n--- hybrid ---")
print(tokenizer.decode(hybrid[0], skip_special_tokens=True)[:400])


In [ ]:
import matplotlib.pyplot as plt

names = [row[0] for row in rows]
nlls = [row[1] for row in rows]
forwards = [row[2] for row in rows]
colors = ["#1d1a16", "#9c4221", "#c47a4a"]

fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.2))
axes[0].bar(names, nlls, color=colors)
axes[0].set_ylabel("teacher NLL")
axes[0].set_title("Closer to the teacher")
axes[1].bar(names, forwards, color=colors)
axes[1].set_ylabel("forwards")
axes[1].set_title("Work per sequence")
axes[2].plot(range(1, len(per_level) + 1), per_level, color="#9c4221", marker="o")
axes[2].set_ylim(0, 1)
axes[2].set_xlabel("level")
axes[2].set_ylabel("code accuracy")
axes[2].set_title("Teacher-forced codes")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()


## How to read a failure

The repair-rule check and the forward count are tests of the hybrid, not of training quality. They should pass on any checkpoint this training notebook wrote.

The NLL check fails when the repair did not pull the draft toward the teacher. A repair rate near 100% means the threshold rejected the draft and the hybrid number is mostly one teacher rewrite of a bad sequence. A reconstruction near chance means the quantizer did not learn. Train longer, or lower `TAU` in the training notebook, before treating the quality numbers as a comparison.
